# Visión parcial de 5×5: 2 secuencias con recompensa

Respecto del 5×5 original, solo cambia el muestreo del buffer: `REWARD_SLICES = 2`. La entropía conserva su valor original (`ACT_ENTROPY = 3e-4`).

**Qué queremos probar.** El 5×5 original terminó consiguiendo cero galletas en evaluación y dejó de visitar el botón. Este experimento intenta aprovechar mejor los pocos éxitos que consiga: de las 16 secuencias de cada batch, 2 se eligen alrededor de un paso con recompensa y las otras 14 se muestrean como siempre. Antes de la primera recompensa, el muestreo sigue siendo el original. El sampler no añade recompensas al ambiente.

**Comparación de esta noche:**

| Experimento | `buffer.reward_slices` | `model.act_entropy` |
|---|---:|---:|
| 5×5 original | 0 | 0.0003 |
| 5×5 sampler | 2 | 0.0003 |
| 5×5 sampler + entropía | 2 | 0.003 |

Las dos pruebas nuevas solo difieren entre sí en el coeficiente de entropía. Se mantienen la tarea `cookie_partial_hall5_view5`, la vista 5×5, 500.000 pasos, `TRAIN_RATIO = 128`, imagen 64×64, decoder (`REP_LOSS = "dreamer"`), buffer de 500.000 en RAM, semilla 0 y evaluación cada 25.000 pasos con 5 partidas. La memoria, las acciones y la recompensa del ambiente son las originales.

**Runs nuevos.** La carpeta termina en `_sampler2`. La primera ejecución parte desde cero; repetir este notebook retoma únicamente su propia carpeta. Conserva ese sufijo al volver otro día. Para usar vista 3×3, cambia `VISTA = 3` en la sección 6 antes de empezar; la tarea y la carpeta cambian con ella. Mantén la misma vista en ambas pruebas si quieres aislar el efecto de la entropía.

**Para dejarlo de noche.** Selecciona L4 GPU y ejecuta las secciones 1 a 10. La sección 8 mide la velocidad con estos cambios; su estimación no incluye los guardados del buffer ni las evaluaciones. `APAGAR_AL_TERMINAR = True` guarda, sincroniza Drive y libera la sesión al terminar correctamente. Cada notebook necesita su propio entorno de ejecución si quieres correrlos a la vez. Antes de dejarlo, verifica que la sección 10 haya empezado a mostrar pasos sin errores.

**Para revisar mañana.** Conecta una sesión (CPU basta para los gráficos), ejecuta las secciones 2, 3, 6 y 11. La sección 6 define `LOGDIR`; no entrena. Para seguir entrenando, ejecuta 1 a 6 y 10. Si se apagó al terminar, la sección 11 se ejecuta en la sesión nueva.

**Cómo interpretar.** Compara `episode/eval_score` con el 5×5 original y entre ambas pruebas. Más entropía o más casillas visitadas sirven para seguir el diagnóstico; el éxito requiere galletas en evaluaciones repetidas. La vista parcial sigue exigiendo memoria y estos cambios no garantizan que la aprenda. La recompensa imaginada puede subir por el muestreo sin que mejore el comportamiento: verifica siempre la evaluación y el video.

**Fundamento.** Dreamer usa regularización de entropía para favorecer exploración; el coeficiente de referencia es `3e-4`. El aumento a `3e-3` es nuestra prueba, no una recomendación validada del artículo: [DreamerV3, aprendizaje del actor](https://arxiv.org/html/2301.04104v2). El sampler ya está implementado en `buffer.py` de este repo.


## 1. Revisar la GPU y la RAM

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import os

ram = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9
print(f"RAM: {ram:.0f} GB")
if ram < 20:
    print("OJO: el buffer de este run ocupa ~11 GB de RAM. Usa la L4 (viene con 53 GB) o activa High-RAM.")

## 2. Montar Drive

Aquí quedan el checkpoint, el buffer y las métricas, para que sobrevivan al corte de sesión.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 3. Clonar el repo (o actualizarlo si ya está)

In [ ]:
%%bash
if [ -d /content/r2dreamer-cookie ]; then
  cd /content/r2dreamer-cookie && git pull --ff-only
else
  git clone https://github.com/FabriRandon/r2dreamer-cookie.git /content/r2dreamer-cookie
fi

## 4. Instalar

r2dreamer necesita Python 3.11 y Colab suele traer una versión más nueva, así que se crea un entorno aparte con `uv`. Toma unos minutos y hay que repetirlo en cada sesión nueva.

In [ ]:
%%bash
pip install -q uv
cd /content/r2dreamer-cookie
uv venv --python 3.11 .venv
uv pip install --python .venv/bin/python -e ".[cookie]" 2>&1 | tail -3

## 5. Comprobar que todo quedó bien

In [ ]:
%%bash
cd /content/r2dreamer-cookie
.venv/bin/python - <<'EOF'
import torch
from envs.cookie import Cookie
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO HAY GPU")
a = torch.randn(256, 256, device="cuda", dtype=torch.float16)
print("float16 ok:", bool(torch.isfinite((a @ a).sum())))
env = Cookie("partial_hall5_view5", size=(64, 64))
obs = env.reset()
grid = env._env.unwrapped
print("mapa:", grid.height, "x", grid.width, "| partida de", grid.max_steps, "pasos (deberían ser 12 x 17 y 816)")
print("vista:", grid.agent_view_size, "x", grid.agent_view_size, "(debería ser 5 x 5)")
print("galleta ok:", obs["image"].shape, "| acciones:", env.action_space.n, "(deberían ser 3)")
print("métricas nuevas:", sorted(k for k in obs if k.startswith("log_")))
EOF

## 6. Configurar el run

**No cambies `VISTA`, `TAREA`, `IMAGEN`, `REP_LOSS` ni `BUFFER` a mitad del run:** el checkpoint quedó hecho con esos valores y al retomar no calzaría. Si quieres probar otros, usa otro `LOGDIR`.

- `PASOS` y `TRAIN_RATIO` sí se pueden cambiar entre sesiones (deteniéndolo con el botón). Por ejemplo, si a los 500.000 todavía está mejorando, sube `PASOS` y sigue.
- `GUARDAR_BUFFER_CADA`: el modelo se guarda cada 10.000 pasos, pero el buffer (~11 GB) solo cada 100.000, porque cada guardado lo escribe completo. Si la sesión se corta sola, se pierden como mucho los datos juntados desde el último guardado del buffer, y se vuelven a juntar. Si lo detienes tú (celda 10), se guarda todo.
- `PARAR_EN_PASO`: para que se detenga solo en un paso y guarde todo, por ejemplo `200_000` para revisarlo ahí. Después se sigue con `None` o con un paso mayor.
- `APAGAR_AL_TERMINAR`: con `True`, la sesión se apaga sola al terminar o al llegar a `PARAR_EN_PASO`, y deja de gastar unidades. No se apaga si lo detienes tú.

- `REWARD_SLICES`: 2 secuencias por batch alrededor de una recompensa, cuando existe alguna. No garantiza descubrir la primera galleta.
- `ACT_ENTROPY`: conserva el valor propio de este experimento al retomar.
- `EXPERIMENTO`: identifica esta prueba en Drive. No lo cambies al retomar; cada prueba usa una carpeta nueva.


In [ ]:
CASO = "base"  # El actor sigue juntando los datos
EXPERIMENTO = "sampler2"
REWARD_SLICES = 2  # de las 16 secuencias, 2 incluyen una recompensa si ya hay alguna
ACT_ENTROPY = 3e-4  # coeficiente del incentivo a mantener acciones variadas

VISTA = 5  # casillas por lado que ve el agente
TAREA = f"cookie_partial_hall5_view{VISTA}"  # visión parcial, pasillos de 5; apretar el botón cambia la galleta de esquina
PASOS = 500_000
TRAIN_RATIO = 128
IMAGEN = 64  # píxeles por lado
REP_LOSS = "dreamer"  # con decoder
SEMILLA = 0
BUFFER = 500_000  # todo el run, en la RAM
GUARDAR_BUFFER_CADA = 100_000
EVALUAR_CADA = 25_000
PARTIDAS_EVAL = 5

# Para que se detenga solo en un paso y siga otro día, por ejemplo 200_000. None: hasta el final.
PARAR_EN_PASO = None
# Apaga la sesión cuando el entrenamiento termina o llega a PARAR_EN_PASO.
APAGAR_AL_TERMINAR = True

LOGDIR = f"/content/drive/MyDrive/galleta/semana15/{TAREA}_s{SEMILLA}_{CASO}_{EXPERIMENTO}"

OPCIONES = (
    f"env=cookie env.task={TAREA} env.train_ratio={TRAIN_RATIO} 'env.size=[{IMAGEN},{IMAGEN}]' "
    f"seed={SEMILLA} model.rep_loss={REP_LOSS} trainer.random_policy={CASO == 'random'} "
    f"trainer.video_pred_log=True buffer.max_size={BUFFER} buffer.storage_device=cpu "
    f"buffer.reward_slices={REWARD_SLICES} model.act_entropy={ACT_ENTROPY}"
)
print(LOGDIR)
print(OPCIONES)
from pathlib import Path

if (Path(LOGDIR) / "latest.pt").exists():
    print("Se retomará el checkpoint de este experimento.")
else:
    print("Experimento nuevo: empieza desde cero.")


## 7. Ver en qué va el run

Dice en qué paso quedó el run guardado en Drive, desde qué paso es el buffer y cómo vienen las últimas partidas comparadas con un agente al azar. Córrela antes de entrenar en una sesión nueva para saber desde dónde va a seguir. Si dice que todavía no hay nada guardado, es un run nuevo.

In [ ]:
!cd /content/r2dreamer-cookie && python3 progress.py --logdir "{LOGDIR}"

## 8. Prueba corta (solo la primera vez)

Corre 1.500 pasos en el disco local con la misma configuración, incluido el buffer de 500.000 en la RAM, pero sin guardarlo. Comprueba que todo funcione y estima cuánto va a tardar el run. Toma unos 10 a 15 minutos.

In [ ]:
!rm -rf /content/prueba_vista5_sampler2
!cd /content/r2dreamer-cookie && timeout 1800 .venv/bin/python -u train.py {OPCIONES} \
    env.steps=1500 env.eval_episode_num=1 trainer.update_log_every=500 trainer.save_every=1e9 \
    trainer.save_buffer=False logdir=/content/prueba_vista5_sampler2 > /content/prueba_vista5_sampler2.log 2>&1; \
    tail -n 5 /content/prueba_vista5_sampler2.log

In [ ]:
import json

lineas = [json.loads(l) for l in open("/content/prueba_vista5_sampler2/metrics.jsonl")]
fps = [l["fps/fps"] for l in lineas if l.get("fps/fps", 0) > 0]
print("evaluación ok:", any("episode/eval_score" in l for l in lineas))
print("métricas nuevas ok:", any("episode/eval_cells" in l for l in lineas))
if not fps:
    print("No alcanzó a medir la velocidad. Revisa el final de /content/prueba_vista5_sampler2.log (arriba).")
else:
    v = sum(fps) / len(fps)
    horas = PASOS / v / 3600
    print(f"{v:.1f} pasos por segundo → {PASOS:,} pasos en unas {horas:.0f} horas "
          f"(~{1.54 * horas:.0f} unidades en una L4), más los guardados del buffer y las evaluaciones")

## 9. Abrir TensorBoard

Ábrelo **antes** de entrenar: mientras la celda de entrenamiento corre, Colab no deja ejecutar otras, pero TensorBoard se sigue actualizando (botón de recargar arriba a la derecha).

En la pestaña **Scalars** (entre paréntesis, lo que hace un agente al azar):

- `episode/score`: galletas por partida mientras entrena (~0,05). Es lo que graficaba Nano.
- `episode/presses`: veces que aprieta el botón por partida (~2,7; el azar ni lo toca en 2 de cada 3 partidas).
- `episode/cells`: casillas distintas que visita por partida (~38).
- `episode/button_room`: pasos que pasa en el cuarto del botón (~155 de los 816).
- `episode/eval_score`: galletas por partida del actor sin azar, cada 25.000 pasos.
- `train/rew`: la recompensa que predice el modelo al soñar. Si se queda en ~0,00001, el modelo cree que nunca hay galleta.
- `train/action_entropy`: 1,099 es azar puro; si baja, el actor empezó a preferir algo.

En la pestaña **Images**:

- `train_video`: la última partida de entrenamiento completa, tal como la ve el agente.
- `open_loop`: arriba lo real, al medio lo que predice el modelo y abajo la diferencia. Los primeros 5 cuadros los reconstruye mirando la imagen; el resto los **imagina** solo con las acciones. Como ve solo lo que tiene delante, la galleta aparece en la imagen recién cuando la tiene a la vista. Si el modelo la entendió, al imaginar la hace aparecer cuando el agente llega a la esquina donde está.

In [ ]:
%load_ext tensorboard
from tensorboard import notebook

notebook.start(f'--logdir "{LOGDIR}"')

## 10. Entrenar (o seguir)

Arriba aparece una barra de progreso con el tiempo que falta, y debajo va corriendo el log. Al retomar, cargar el buffer desde Drive tarda unos minutos antes de que la barra aparezca.

**Para detenerlo:** usa el botón de detener de esta celda. El entrenamiento termina el paso en que va, guarda el modelo y el buffer en Drive (puede tardar varios minutos) y avisa con `Listo, quedó guardado`. No cierres la pestaña antes de ese mensaje.

**Para seguir otro día:** corre las celdas 1 a 6 y después esta. Sigue exactamente desde el paso en que lo detuviste.

**Si la sesión se corta sola** (desconexión, límite de Colab): sigue igual, desde el último guardado. El modelo se guarda cada 10.000 pasos y el buffer cada 100.000; el log avisa qué tramo de datos se perdió, y se vuelve a juntar.

Para dejarlo corriendo de noche: deja la pestaña abierta y que el PC no entre en suspensión (la pantalla sí se puede apagar). Cada guardado reemplaza el buffer anterior, que queda en la papelera de Drive; puedes vaciarla al final.

In [ ]:
import re
import subprocess

from tqdm.auto import tqdm

parar = f" trainer.stop_at={PARAR_EN_PASO}" if PARAR_EN_PASO else ""
comando = (
    # exec deja a Python en lugar de la shell, para que el botón de detener le llegue a él.
    f"exec .venv/bin/python -u train.py {OPCIONES} env.steps={PASOS} env.eval_episode_num={PARTIDAS_EVAL} "
    f"trainer.eval_every={EVALUAR_CADA} trainer.update_log_every=2500 trainer.save_every=1e4 "
    f'trainer.save_buffer_every={GUARDAR_BUFFER_CADA}{parar} logdir="{LOGDIR}"'
)
proceso = subprocess.Popen(comando, shell=True, cwd="/content/r2dreamer-cookie", text=True, bufsize=1,
                           stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
barra = None
detenido = False


def mostrar(linea):
    global barra
    paso = re.match(r"\[(\d+)\]", linea)
    if paso:
        paso = min(int(paso.group(1)), PASOS)
        if barra is None:
            # Parte desde el paso en que va, por si el run se retomó a mitad de camino.
            barra = tqdm(total=PASOS, initial=paso, unit="paso", smoothing=0, desc=EXPERIMENTO)
        barra.update(paso - barra.n)
        if "eval_score" in linea or "Saved the replay buffer" in linea:
            barra.set_postfix_str("")
        if " progress " in linea:
            return  # la barra ya muestra el avance
    if barra is not None and linea.startswith("Evaluating"):
        barra.set_postfix_str("evaluando")
    if barra is not None and "Saving the replay buffer" in linea:
        barra.set_postfix_str("guardando el buffer")
    print(linea, end="")


try:
    for linea in proceso.stdout:
        mostrar(linea)
except KeyboardInterrupt:
    detenido = True
    # train.py recibe la señal, termina el paso en que va y guarda todo antes de salir.
    proceso.terminate()
    print("\nDeteniendo: se está guardando el modelo y el buffer en Drive. Puede tardar varios minutos; "
          "no cierres la pestaña hasta que diga que quedó guardado.")
    while proceso.poll() is None:
        try:
            for linea in proceso.stdout:
                print(linea, end="")
        except KeyboardInterrupt:
            print("Ya se está guardando; espera a que termine.")
finally:
    if barra is not None:
        barra.close()
proceso.wait()
if proceso.returncode != 0:
    print(f"El entrenamiento terminó con un error (código {proceso.returncode}). Revisa las últimas líneas de arriba.")
elif detenido:
    print("Listo, quedó guardado. Para seguir otro día: celdas 1 a 6 y después esta.")

if APAGAR_AL_TERMINAR and not detenido and proceso.returncode == 0:
    from google.colab import drive, runtime

    # Drive sube los archivos con atraso: esto espera a que todo quede guardado antes de apagar.
    print("Guardando todo en Drive y apagando la sesión para que deje de gastar unidades...")
    drive.flush_and_unmount()
    runtime.unassign()

## 11. Ver el progreso

Lo mismo que la celda 7, más los gráficos: galletas, veces que aprieta el botón, casillas visitadas, pasos en el cuarto del botón, la recompensa que predice el modelo y la entropía del actor. La línea gris es un agente al azar.

Mientras la celda 10 corre, esto mismo se ve desde `monitor.ipynb` en otra pestaña.

Si usas `monitor.ipynb`, pega allí el `LOGDIR` que imprime la sección 6 de este notebook; el selector original del monitor apunta al run sin sampler. Tras el apagado automático, vuelve a montar Drive y ejecutar la sección 6 antes de graficar.


In [ ]:
!cd /content/r2dreamer-cookie && python3 progress.py --logdir "{LOGDIR}" --out /content/progreso_vista5_sampler2.png

from IPython.display import Image, display

display(Image("/content/progreso_vista5_sampler2.png"))

## 12. Ver al agente jugar

Carga el agente desde Drive (`latest.pt`, el último guardado) y lo pone a jugar una partida completa con lo que aprendió, sin azar, igual que en la evaluación. El video queda en la carpeta del run.

- A la izquierda está el mapa, con lo que ve el agente iluminado, y a la derecha la imagen que recibe.
- Arriba van el paso, las galletas que lleva y la jugada que eligió.

Con `SEMILLA_EPISODIO` cambian las esquinas donde aparece la galleta. Tarda uno o dos minutos.

In [ ]:
SEMILLA_EPISODIO = 0
VIDEO = f"{LOGDIR}/agente_episodio{SEMILLA_EPISODIO}.mp4"

!cd /content/r2dreamer-cookie && .venv/bin/python watch.py --logdir "{LOGDIR}" --out "{VIDEO}" \
    --seed {SEMILLA_EPISODIO} 2>&1 | tail -n 1

from IPython.display import Video
Video(VIDEO, embed=True, width=760)